# Apple Leaf Disease **Detection** with YOLO (PlantDoc)

This notebook adds an explicit **object-detection** component to the thesis, using bounding boxes from the **PlantDoc** detection dataset for the three apple classes (Apple Scab Leaf, Apple rust leaf, Apple leaf/healthy).

**Run order:** Cell 1 (install) → Cell 2 (upload the two .py files) → Cell 3 (prepare data) → Cell 4 (train + eval) → Cell 5 (zip results to send back).

Use a **GPU runtime** (Runtime → Change runtime type → GPU).

## 1) Install dependencies

In [1]:
import subprocess, sys, torch, ultralytics
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'ultralytics'], check=True)
print('ultralytics', ultralytics.__version__, '| CUDA:', torch.cuda.is_available())

ultralytics 8.4.90 | CUDA: True


## 2) Add the two scripts
Upload `prepare_plantdoc_yolo.py` and `train_eval_yolo.py` (from the folder I sent) using the file panel on the left, **or** run the cell below and pick them.

In [2]:
# Running locally: scripts already present in this folder (not Colab)
import os
for f in ["prepare_plantdoc_yolo.py", "train_eval_yolo.py"]:
    assert os.path.exists(f), f"missing {f}"
print("scripts found locally, skipping Colab upload")

scripts found locally, skipping Colab upload


## 3) Prepare the PlantDoc apple detection dataset
Clones PlantDoc, converts VOC boxes to YOLO format, keeps only apple images, writes `apple_det/data.yaml`.

In [3]:
import subprocess, sys
subprocess.run([sys.executable, 'prepare_plantdoc_yolo.py', '--out', './apple_det'], check=True)

CompletedProcess(args=['C:\\Users\\cy\\AppData\\Local\\Programs\\Python\\Python310\\python.exe', 'prepare_plantdoc_yolo.py', '--out', './apple_det'], returncode=0)

## 4) Train + evaluate YOLO
~100 epochs. If `yolo11s.pt` errors on your version, change `--model` to `yolov8s.pt`.

In [4]:
import subprocess, sys
subprocess.run([sys.executable, 'train_eval_yolo.py', '--data', './apple_det/data.yaml',
                '--model', 'yolo11s.pt', '--epochs', '100', '--imgsz', '640',
                '--batch', '16', '--name', 'apple_yolo'], check=True)

CompletedProcess(args=['C:\\Users\\cy\\AppData\\Local\\Programs\\Python\\Python310\\python.exe', 'train_eval_yolo.py', '--data', './apple_det/data.yaml', '--model', 'yolo11s.pt', '--epochs', '100', '--imgsz', '640', '--batch', '16', '--name', 'apple_yolo'], returncode=0)

In [5]:
# quick look at the final metrics table
import pandas as pd
pd.read_csv('runs/apple_yolo/metrics_summary.csv')

,class,precision,recall,mAP@0.5,mAP@0.5:0.95
0,ALL,0.8580,0.8120,0.9268,0.7132
1,Apple_Scab_Leaf,0.8811,0.6923,0.8592,0.6789
2,Apple_rust_leaf,0.8224,0.8436,0.9429,0.7392
3,Apple_leaf,0.8705,0.9000,0.9783,0.7215


## 5) Zip everything to send back
Download `apple_yolo_results.zip` and send it to me — I will fold the numbers and figures into the thesis and paper.

In [1]:
import shutil, glob, os
os.makedirs('send_back', exist_ok=True)
for pat in ['runs/apple_yolo/metrics_summary.csv','runs/apple_yolo/metrics_summary.json',
            'runs/detect/runs/apple_yolo/results.png','runs/detect/runs/apple_yolo/results.csv',
            'runs/detect/runs/apple_yolo/confusion_matrix.png','runs/detect/runs/apple_yolo/confusion_matrix_normalized.png',
            'runs/detect/runs/apple_yolo/*PR_curve.png','runs/detect/runs/apple_yolo/*F1_curve.png',
            'runs/detect/runs/apple_yolo/val_batch0_pred.jpg','runs/detect/runs/apple_yolo/val_batch0_labels.jpg',
            'runs/detect/runs/apple_yolo_val/confusion_matrix.png','runs/detect/runs/apple_yolo_val/*PR_curve.png',
            'runs/detect/runs/apple_yolo_preds/*']:
    for f in glob.glob(pat):
        shutil.copy(f, 'send_back/'+os.path.basename(f))
shutil.make_archive('apple_yolo_results', 'zip', 'send_back')
print('apple_yolo_results.zip ready in', os.getcwd())

apple_yolo_results.zip ready in D:\ADDC new\YOLO\yolo_apple_detection
